# Generating CMB Spectra and Maps

Produces four simulation sets for ML/statistics applications:

| Set | Posterior (tight) | Broad (LHC) | Total |
|-----|-------------------|-------------|-------|
| **Planck ΛCDM** | 500 — Planck 2018 ±5σ | 5000 — 6-D LHC | 5500 |
| **w0wa** | 100 — DESI 2024 ±3σ | 1000 — 8-D LHC | 1100 |
| **Oscillations** | — | 1000 — 8-D LHC | 1000 |
| **Early Dark Energy** | — | 1000 — 9-D LHC | 1000 |

**Complexity knobs** let you toggle each pipeline stage independently:
```
instrument noise → galactic mask → lensing → polarization (Q/U/E/B)
```

**Pipeline per realisation:**
1. CAMB → lensed (or unlensed) Dℓ spectra
2. Dℓ → Cℓ (truncated at ℓ_max = 3·nside − 1, the map band limit)
3. Synthesise T, Q, U on HEALPix sphere
4. Joint spin-2 beam smoothing (5 arcmin FWHM)
5. Add white instrument noise in the map domain (optional)
6. Apply apodised Planck galactic mask (optional) — common Int mask for T, common Pol mask for Q/U
7. Derive pseudo-E, B from masked spin-2 maps (optional)
8. Save float32 FITS + metadata CSV

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit this cell only
# ═══════════════════════════════════════════════════════════════════════════════

# ── Complexity knobs (succession: noise → mask → lensing → polarization) ──────
# Phase 1 Stage 1 starts CLEAN and temperature-only. Turn these on one at a time
# as you climb the complexity ladder (noise → mask → lensing → polarization).
INCLUDE_NOISE         = True   # add white instrument noise in the MAP domain
INCLUDE_GALACTIC_MASK = False  # apply Planck galactic mask to maps
USE_LENSED_SPECTRA    = False  # use lensed CAMB spectra (BB non-zero from lensing)
INCLUDE_POLARIZATION  = False  # save T,Q,U,E,B (False → T only)

# ── Simulation counts ─────────────────────────────────────────────────────────
# Planck LCDM gets the full run; w0wa, oscillations and EDE are scaled down.
N_POSTERIOR_PLANCK = 500    # Planck 2018 posterior-chain draws, extra on top of N_BROAD_PLANCK
N_POSTERIOR_W0WA   = 100    # DESI 2024 posterior-chain draws, extra on top of N_BROAD_W0WA
N_BROAD_PLANCK     = 5000   # LHC broad draws for the Planck LCDM set
N_BROAD_W0WA       = 1000   # LHC broad draws for the w0wa set
N_OSC              = 1000   # LHC broad draws for the oscillations set
N_EDE              = 1000   # LHC broad draws for the early dark energy set

# ── Instrument noise level (used only when INCLUDE_NOISE) ─────────────────────
# White noise added per pixel AFTER the beam: sigma_pix = level / pixel_size[arcmin].
NOISE_UKARCMIN_T = 40.0              # representative Planck-like T noise [μK·arcmin]
NOISE_UKARCMIN_P = 40.0 * 2 ** 0.5  # polarization noise is sqrt(2) higher

# ── Seeds ─────────────────────────────────────────────────────────────────────
BASE_SEED = 314100   # base map seed
LHC_SEED  = 42       # LHC sampler seed (fixed for reproducibility)

# Per-set offsets so the cosmic-variance AND noise realisations are INDEPENDENT
# across the four sets (no shared seed at matching indices).
SEED_OFFSET       = {'planck': 0, 'w0wa': 1_000_000, 'oscillations': 2_000_000,
                     'ede': 3_000_000}
NOISE_SEED_OFFSET = 7_000_000    # keeps the noise RNG stream clear of all map seeds

# ── HEALPix resolution ────────────────────────────────────────────────────────
nside    = 256
lmax_map = 3 * nside - 1   # band limit of an nside map; synfast cannot exceed this


In [12]:
import os
import csv
import math
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import camb
import pandas as pd
from scipy.stats import qmc
from collections import namedtuple
from tqdm.notebook import tqdm
from getdist import loadMCSamples

from skysimulation import (
    generate_camb_power_spectra,
    add_noise_spectrum,
    apply_galactic_mask,
    new_resolution_mask,
    PK as _PK,
)

print(f'CAMB version: {camb.__version__}')
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

CAMB version: 1.6.6


In [ ]:
data_dir  = '/nvme/h/lchristodoulou/cmb/SkySimulation/data/Planck/'
mask_path_int = os.path.join(data_dir, 'COM_Mask_CMB-common-Mask-Int_2048_R3.00.fits')
mask_path_pol = os.path.join(data_dir, 'COM_Mask_CMB-common-Mask-Pol_2048_R3.00.fits')

# Config-tag the output root so different complexity-knob combinations (Cell 2)
# never collide: a T-only baseline run and a later polarization run (or any other
# knob flip) land in separate directories instead of overwriting each other's
# FITS/metadata.
_config_tag = (
    f"noise{'T' if INCLUDE_NOISE else 'F'}"
    f"_mask{'T' if INCLUDE_GALACTIC_MASK else 'F'}"
    f"_lens{'T' if USE_LENSED_SPECTRA else 'F'}"
    f"_pol{'T' if INCLUDE_POLARIZATION else 'F'}"
)

_maps_base = os.path.join('/nvme/h/lchristodoulou/data_p318/simulations/', _config_tag)
out_planck = os.path.join(_maps_base, 'planck')        # Set 1: ΛCDM posterior + broad
out_w0wa   = os.path.join(_maps_base, 'w0wa')          # Set 2: w0wa posterior + broad
out_osc    = os.path.join(_maps_base, 'oscillations')  # Set 3: oscillations broad
out_ede    = os.path.join(_maps_base, 'ede')           # Set 4: early dark energy broad

for d in [out_planck, out_w0wa, out_osc, out_ede]:
    os.makedirs(d, exist_ok=True)
print(f'Output directories ready under: {_maps_base}')

## 1. Load Planck Observational Data

Planck 2018 binned power spectra $D_\ell = \frac{\ell(\ell+1)}{2\pi}C_\ell$ ($\mu K^2$) and covariance matrices used for noise sampling.

In [14]:
d = np.loadtxt(os.path.join(data_dir, 'COM_PowerSpect_CMB-TT-full_R3.01.txt'))
ls_TT, dl_TT = d[:, 0], d[:, 1]
round_ls_TT = np.round(ls_TT).astype(int)

d = np.loadtxt(os.path.join(data_dir, 'COM_PowerSpect_CMB-TE-full_R3.01.txt'))
ls_TE, dl_TE = d[:, 0], d[:, 1]
round_ls_TE = np.round(ls_TE).astype(int)

d = np.loadtxt(os.path.join(data_dir, 'COM_PowerSpect_CMB-EE-full_R3.01.txt'))
ls_EE, dl_EE = d[:, 0], d[:, 1]
round_ls_EE = np.round(ls_EE).astype(int)

cov_tt = np.loadtxt(os.path.join(data_dir, 'dlstt_cov_matx(mcmc).csv'), delimiter=',')
cov_te = np.loadtxt(os.path.join(data_dir, 'dlste_cov_matx(mcmc).csv'), delimiter=',')
cov_ee = np.loadtxt(os.path.join(data_dir, 'dlsee_cov_matx(mcmc).csv'), delimiter=',')

# lmax constrained by the shortest Planck spectrum (TE/EE end at ell≈1996)
lmax_synfast = min(int(round_ls_TT.max()), int(round_ls_TE.max()), int(round_ls_EE.max()))
print(f'TT: {len(ls_TT)} bins  TE: {len(ls_TE)} bins  EE: {len(ls_EE)} bins')
print(f'lmax_synfast = {lmax_synfast}')

TT: 2507 bins  TE: 1995 bins  EE: 1995 bins
lmax_synfast = 1996


## 2. Helper Functions

All helpers read the configuration knobs set in Cell 2.

In [ ]:
_CMBSpectra = namedtuple('CMBSpectra', ['ell', 'tt', 'te', 'ee', 'bb', 'theta_star'])


def _set_dark_energy(params, w0, wa, ede_params):
    """Attach the DE sector: AxionEffectiveFluid if ede_params given, else PPF(w0,wa).

    ede_params, when given, is (w_n, fde_zc, zc, theta_i) and fully REPLACES the
    w0/wa dark energy sector (AxionEffectiveFluid models the whole DE fluid —
    early rolling-axion behaviour plus the late-time approach to w=-1 — so w0/wa
    are not meaningful alongside it).
    """
    if ede_params is not None:
        w_n, fde_zc, zc, theta_i = ede_params
        params.DarkEnergy = camb.dark_energy.AxionEffectiveFluid()
        params.DarkEnergy.set_params(w_n=w_n, fde_zc=fde_zc, zc=zc, theta_i=theta_i)
    else:
        params.DarkEnergy = camb.dark_energy.DarkEnergyPPF()
        params.DarkEnergy.set_params(w=w0, wa=wa)


def _lensed_spectra(H0, ombh2, omch2, mnu, omk, tau, As, ns,
                    lmax=2507, halofit_version='mead2020_feedback',
                    w0=-1.0, wa=0.0,
                    custom_PK=False, amp=0, freq=0, wid=1, centre=0.05, phase=0,
                    ede_params=None):
    """Lensed CMB Dl spectra (μK²). DarkEnergyPPF handles phantom crossing."""
    params = camb.set_params(
        H0=H0, ombh2=ombh2, omch2=omch2, mnu=mnu, omk=omk, tau=tau,
        As=As, ns=ns, lmax=lmax, halofit_version=halofit_version,
    )
    _set_dark_energy(params, w0, wa, ede_params)
    if custom_PK:
        params.set_initial_power_function(
            _PK, args=(As, ns, amp, freq, wid, centre, phase),
            effective_ns_for_nonlinear=ns,
        )
    else:
        params.InitPower.set_params(As=As, ns=ns)
    results    = camb.get_results(params)
    lensed     = results.get_cmb_power_spectra(params, CMB_unit='muK')['lensed_scalar']
    ell        = np.arange(len(lensed))
    theta_star = results.get_derived_params()['thetastar']
    return _CMBSpectra(ell, lensed[:, 0], lensed[:, 3], lensed[:, 1], lensed[:, 2], theta_star)


def _unlensed_spectra(H0, ombh2, omch2, mnu, omk, tau, As, ns,
                      lmax=2507, w0=-1.0, wa=0.0,
                      custom_PK=False, amp=0, freq=0, wid=1, centre=0.05, phase=0,
                      ede_params=None):
    """Unlensed CMB Dl spectra (BB = 0 identically). Mirrors _lensed_spectra so
    that w0/wa (or the EDE sector) are respected regardless of USE_LENSED_SPECTRA."""
    params = camb.set_params(
        H0=H0, ombh2=ombh2, omch2=omch2, mnu=mnu, omk=omk, tau=tau,
        As=As, ns=ns, lmax=lmax,
    )
    _set_dark_energy(params, w0, wa, ede_params)
    if custom_PK:
        params.set_initial_power_function(
            _PK, args=(As, ns, amp, freq, wid, centre, phase),
            effective_ns_for_nonlinear=ns,
        )
    else:
        params.InitPower.set_params(As=As, ns=ns)
    results    = camb.get_results(params)
    unlensed   = results.get_cmb_power_spectra(params, CMB_unit='muK')['unlensed_scalar']
    ell        = np.arange(len(unlensed))
    bb         = np.zeros(len(ell))
    theta_star = results.get_derived_params()['thetastar']
    return _CMBSpectra(ell, unlensed[:, 0], unlensed[:, 3], unlensed[:, 1], bb, theta_star)


def generate_spectra(H0, ombh2, omch2, mnu, omk, tau, As, ns,
                     lmax=2507, w0=-1.0, wa=0.0,
                     custom_PK=False, amp=0, freq=0, wid=1, centre=0.05, phase=0,
                     ede_params=None):
    """Dispatch to lensed or unlensed CAMB spectra based on USE_LENSED_SPECTRA.

    ede_params, when given as (w_n, fde_zc, zc, theta_i), switches the DE sector
    to camb.dark_energy.AxionEffectiveFluid and overrides w0/wa (see
    _set_dark_energy).
    """
    if USE_LENSED_SPECTRA:
        return _lensed_spectra(H0, ombh2, omch2, mnu, omk, tau, As, ns,
                               lmax=lmax, w0=w0, wa=wa,
                               custom_PK=custom_PK, amp=amp, freq=freq,
                               wid=wid, centre=centre, phase=phase,
                               ede_params=ede_params)
    else:
        return _unlensed_spectra(H0, ombh2, omch2, mnu, omk, tau, As, ns,
                                 lmax=lmax, w0=w0, wa=wa,
                                 custom_PK=custom_PK, amp=amp, freq=freq,
                                 wid=wid, centre=centre, phase=phase,
                                 ede_params=ede_params)


def dl_to_cl(dl_camb):
    """
    Convert a CAMB D_ell array (μK², indexed from ell=0) to a C_ell array of
    length lmax_map+1 for synfast. No Planck band-power round-trip and no noise
    here — instrument noise is added in the map domain (see make_and_save_maps).
    """
    ell = np.arange(lmax_map + 1, dtype=float)
    dl  = np.zeros(lmax_map + 1)
    k   = min(len(dl_camb), lmax_map + 1)
    dl[:k] = dl_camb[:k]
    with np.errstate(divide='ignore', invalid='ignore'):
        return np.where(ell > 1, 2 * np.pi * dl / (ell * (ell + 1)), 0.0)


def save_spectra(sp, cl_tt, cl_ee, cl_te, cl_bb, out_dir, tag):
    """Save ell, Cl and Dl arrays as a .npz in a spectra/ subdirectory.

    Always saves TT. Saves EE, TE, BB only when INCLUDE_POLARIZATION is True.
    All arrays are truncated to lmax_map (the map band limit), matching synfast input.
    Load with: data = np.load('spectra_<tag>.npz')
    """
    spectra_dir = os.path.join(out_dir, 'spectra')
    os.makedirs(spectra_dir, exist_ok=True)
    n   = lmax_map + 1
    ell = np.arange(n)
    data = {
        'ell':   ell,
        'cl_tt': cl_tt,
        'dl_tt': sp.tt[:n],
    }
    if INCLUDE_POLARIZATION:
        data.update({
            'cl_ee': cl_ee,
            'dl_ee': sp.ee[:n],
            'cl_te': cl_te,
            'dl_te': sp.te[:n],
            'cl_bb': cl_bb,
            'dl_bb': sp.bb[:n],
        })
    np.savez(os.path.join(spectra_dir, f'spectra_{tag}.npz'), **data)


def make_and_save_maps(cl_tt, cl_ee, cl_bb, cl_te,
                       mask_t, mask_p, nside, seed_i, fwhm_rad, out_dir, tag):
    """
    Synthesise CMB maps and save as float32 FITS.
    Order: signal (synfast) -> beam -> white noise -> mask -> optional E/B.
    Respects INCLUDE_NOISE, INCLUDE_GALACTIC_MASK and INCLUDE_POLARIZATION.
    """
    npix = hp.nside2npix(nside)
    n    = len(cl_tt)

    # Cosmic-variance realisation. synfast draws from the global RNG, so seed it
    # deterministically here (a separate stream from the noise RNG below).
    np.random.seed(seed_i)
    T, Q, U = hp.synfast(
        [cl_tt, cl_ee, cl_bb, cl_te, np.zeros(n), np.zeros(n)],
        nside=nside, new=True, pol=True, lmax=lmax_map,
    )
    T, Q, U = hp.smoothing([T, Q, U], fwhm=fwhm_rad, pol=True)

    if INCLUDE_NOISE:
        # White instrument noise in the map domain, added AFTER the beam.
        rng_noise  = np.random.default_rng(seed_i + NOISE_SEED_OFFSET)
        pix_arcmin = np.degrees(hp.nside2resol(nside)) * 60.0
        sigma_T    = NOISE_UKARCMIN_T / pix_arcmin
        sigma_P    = NOISE_UKARCMIN_P / pix_arcmin
        T = T + rng_noise.normal(0.0, sigma_T, npix)
        Q = Q + rng_noise.normal(0.0, sigma_P, npix)
        U = U + rng_noise.normal(0.0, sigma_P, npix)

    if INCLUDE_GALACTIC_MASK:
        T *= mask_t;  Q *= mask_p;  U *= mask_p

    if INCLUDE_POLARIZATION:
        _, alm_E, alm_B = hp.map2alm([T, Q, U], pol=True)
        E = hp.alm2map(alm_E, nside=nside)
        B = hp.alm2map(alm_B, nside=nside)
        comps = [('T', T), ('Q', Q), ('U', U), ('E', E), ('B', B)]
    else:
        comps = [('T', T)]

    for comp, m in comps:
        hp.write_map(os.path.join(out_dir, f'{comp}_{tag}.fits'),
                     m, dtype=np.float32, overwrite=True)


# ── Apodised masks (computed once, shared by all loops) ───────────────────────
# T uses the common intensity mask; Q/U use the common polarization mask (they
# differ — Pol excludes additional polarized synchrotron/point-source regions).
_mask_hard_int = new_resolution_mask(mask_path_int, target_nside=nside, threshold=0.9)
_mask_hard_pol = new_resolution_mask(mask_path_pol, target_nside=nside, threshold=0.9)
sim_mask_int = np.clip(hp.smoothing(_mask_hard_int, fwhm=np.radians(2.0)), 0.0, 1.0)
sim_mask_pol = np.clip(hp.smoothing(_mask_hard_pol, fwhm=np.radians(2.0)), 0.0, 1.0)
fwhm_rad_sim = np.radians(5.0 / 60)

# ── Fixed parameters shared by all simulation sets ────────────────────────────
mnu       = 0.06    # sum of neutrino masses [eV]
omk       = 0.0     # spatial curvature (flat universe)
tau_fixed = 0.0544  # tau for broad LHC runs while INCLUDE_POLARIZATION=False: TT alone
                     # can't separate tau from As (they enter only as As*exp(-2*tau)), so
                     # varying tau independently there would just add unlearnable label
                     # noise. Once INCLUDE_POLARIZATION=True the low-ell EE reionization
                     # bump breaks that degeneracy, and the LHC draws tau independently
                     # instead (see _lhc_tau below) — tau_fixed is then unused.

# w_n (effective equation of state once the axion field starts oscillating) is not a
# free nuisance parameter of AxionEffectiveFluid: it is fixed by the power n of the
# potential V(phi) ∝ [1-cos(phi/f)]^n via w_n=(n-1)/(n+1), a discrete/theoretical choice
# rather than a continuous one to put a prior on. n=3 -> w_n=1/2 is the standard
# benchmark used across the EDE literature (Poulin, Smith, Karwal & Kamionkowski 2018,
# arXiv:1806.10608; Smith et al. 2020; Hill et al. 2020) — fixed here, not sampled.
EDE_W_N = 0.5

print(f'Apodised masks ready.  f_sky_int = {sim_mask_int.mean():.4f}  '
      f'f_sky_pol = {sim_mask_pol.mean():.4f}')

## 3. Latin Hypercube Samplers

All broad components use `scipy.stats.qmc.LatinHypercube` with fixed seeds for reproducibility.

| Sampler | Dims | Extra params |
|---------|------|--------------|
| `lhc_lcdm_6d` | 6 | τ ∈ [0.02, 0.10]† |
| `lhc_w0wa_8d` | 8 | w0 ∈ [−1.5, −0.5], wa ∈ [−3.0, 1.0], τ ∈ [0.02, 0.10]† |
| `lhc_osc_8d`  | 8 | A_lin ∈ [0.01, 0.06], freq ∈ [20, 40], phase ∈ [0, 2π], τ ∈ [0.02, 0.10]† |
| `lhc_ede_9d`  | 9 | f_EDE ∈ [0, 0.5]‡, theta_i ∈ [0.1, 3.1], log10(zc) ∈ [3, 4.3], τ ∈ [0.02, 0.10]† |

†τ is drawn independently only when `INCLUDE_POLARIZATION=True` — the low-ℓ EE
reionization bump is what lets τ be disentangled from As (in TT alone they only enter
as As·e^{-2τ}). While polarization is off, the τ column is still reserved in the LHC
(so toggling the flag doesn't reshuffle the other columns) but its value is discarded
in favor of `tau_fixed`. See `_lhc_tau` below.

‡floored at 1e-4 rather than the literal 0.0 — CAMB's `AxionEffectiveFluid` hard-crashes
at `fde_zc=0.0` exactly (a native abort, not a catchable Python exception). See
`lhc_ede_9d` docstring.

In [ ]:
def _scale_uniform(u, lo, hi):
    return lo + u * (hi - lo)

def _scale_loguniform(u, lo, hi):
    return np.exp(np.log(lo) + u * (np.log(hi) - np.log(lo)))


def _lhc_tau(u_col):
    """tau column, gated by INCLUDE_POLARIZATION.

    TT alone can't separate tau from As (they enter only as As*exp(-2*tau)), so an
    independent broad draw there would just add unlearnable label noise. Once
    INCLUDE_POLARIZATION=True the low-ell EE reionization bump breaks that
    degeneracy, so tau is drawn independently like every other LHC parameter.
    The LHC always reserves this column (see callers) so the other columns are
    identical regardless of the flag; only which values get returned changes.
    """
    if INCLUDE_POLARIZATION:
        return _scale_uniform(u_col, 0.02, 0.10)
    return np.full(u_col.shape, tau_fixed)


def lhc_lcdm_6d(n, seed=LHC_SEED):
    """
    6-D Latin Hypercube over base ΛCDM parameters + tau.
    Parameter order: omega_b, omega_cdm, H0, ns, As, tau.
    As is log-uniform; others are uniform. tau is gated by INCLUDE_POLARIZATION
    (see _lhc_tau).
    Returns dict of arrays of length n.
    """
    sampler = qmc.LatinHypercube(d=6, seed=seed)
    u = sampler.random(n=n)   # shape (n, 6) in [0, 1)
    return {
        'ombh2':  _scale_uniform(   u[:, 0], 0.018,   0.028),
        'omch2':  _scale_uniform(   u[:, 1], 0.09,    0.15),
        'H0':     _scale_uniform(   u[:, 2], 60.0,    80.0),
        'ns':     _scale_uniform(   u[:, 3], 0.90,    1.10),
        'As':     _scale_loguniform(u[:, 4], 1.5e-9,  2.5e-9),
        'tau':    _lhc_tau(         u[:, 5]),
    }


def lhc_w0wa_8d(n, seed=LHC_SEED + 1):
    """
    8-D Latin Hypercube: 5 ΛCDM params + w0 + wa + tau.
    w0 ∈ [−1.5, −0.5], wa ∈ [−3.0, 1.0] (uniform). tau is gated by
    INCLUDE_POLARIZATION (see _lhc_tau).

    Independent uniform draws over that box put ~3% of points in the region
    w0 + wa > 0, which CAMB's DarkEnergyEqnOfState.validate_params() rejects
    (it implies w(a=0) > 0, unphysical). Those points are resampled in-place
    with plain uniform draws from a dedicated RNG stream until valid, leaving
    the other LHC dimensions untouched.
    """
    sampler = qmc.LatinHypercube(d=8, seed=seed)
    u = sampler.random(n=n)
    w0 = _scale_uniform(u[:, 5], -1.5, -0.5)
    wa = _scale_uniform(u[:, 6], -3.0,  1.0)

    rng = np.random.default_rng(seed + 100_000)
    bad = w0 + wa > 0
    while np.any(bad):
        idx = np.flatnonzero(bad)
        w0[idx] = rng.uniform(-1.5, -0.5, size=idx.size)
        wa[idx] = rng.uniform(-3.0,  1.0, size=idx.size)
        bad = w0 + wa > 0

    return {
        'ombh2':  _scale_uniform(   u[:, 0], 0.018,   0.028),
        'omch2':  _scale_uniform(   u[:, 1], 0.09,    0.15),
        'H0':     _scale_uniform(   u[:, 2], 60.0,    80.0),
        'ns':     _scale_uniform(   u[:, 3], 0.90,    1.10),
        'As':     _scale_loguniform(u[:, 4], 1.5e-9,  2.5e-9),
        'w0':     w0,
        'wa':     wa,
        'tau':    _lhc_tau(u[:, 7]),
    }


def lhc_osc_8d(n, seed=LHC_SEED + 2):
    """
    8-D Latin Hypercube: 4 ΛCDM params (ombh2, omch2, ns, As) + A_lin + freq +
    phase + tau. H0 is FIXED for this set (see Set 3 loop), so it is not
    sampled here. tau is gated by INCLUDE_POLARIZATION (see _lhc_tau).
    A_lin ∈ [0.01, 0.06] (uniform); freq ∈ [20, 40] (uniform, kept float);
    phase ∈ [0, 2π] (uniform).
    Low freq places oscillations at Δℓ ~ 50–100 across the first three acoustic
    peaks (ℓ ~ 200–500) — the cleanly-resolved Phase 1 discrimination regime.
    freq and phase are left continuous to preserve LHC coverage.
    """
    sampler = qmc.LatinHypercube(d=8, seed=seed)
    u = sampler.random(n=n)
    return {
        'ombh2':  _scale_uniform(   u[:, 0], 0.018,  0.028),
        'omch2':  _scale_uniform(   u[:, 1], 0.09,   0.15),
        'ns':     _scale_uniform(   u[:, 2], 0.90,   1.10),
        'As':     _scale_loguniform(u[:, 3], 1.5e-9, 2.5e-9),
        'A_lin':  _scale_uniform(   u[:, 4], 0.01,   0.06),
        'freq':   _scale_uniform(   u[:, 5], 20.0,   40.0),
        'phase':  _scale_uniform(   u[:, 6], 0.0,    2 * np.pi),
        'tau':    _lhc_tau(         u[:, 7]),
    }


def lhc_ede_9d(n, seed=LHC_SEED + 3):
    """
    9-D Latin Hypercube: 5 ΛCDM params (ombh2, omch2, H0, ns, As) + f_EDE +
    theta_i + logzc + tau. H0 is left FREE (unlike Set 3's fixed H0_osc):
    early dark energy's entire point is that raising f_EDE lets H0 float higher
    while still fitting theta_star, so fixing H0 here would hide the very
    degeneracy this set exists to probe.

    f_EDE (fde_zc) ∈ [0, 0.5], theta_i ∈ [0.1, 3.1], log10(zc) ∈ [3, 4.3]
    (all uniform) — the standard EDE prior box (e.g. Poulin et al. 2018;
    Smith et al. 2020; Hill et al. 2020). zc = 10**logzc is passed to CAMB.

    w_n (the fluid's effective equation of state once the field starts
    oscillating) is NOT one of the sampled dimensions: it is set by the power
    n of the axion potential via w_n=(n-1)/(n+1), a discrete choice of
    potential shape rather than a continuous nuisance parameter with its own
    prior in the literature. n=3 (w_n=1/2, EDE_W_N) is fixed for every draw —
    see the note by EDE_W_N above.

    f_EDE is floored at 1e-4 rather than allowed to reach the prior's literal
    lower bound of 0.0: camb.dark_energy.AxionEffectiveFluid hard-crashes
    (native abort — not a catchable Python exception) at fde_zc=0.0 exactly,
    confirmed empirically. LatinHypercube's default scrambling makes landing
    on exact 0.0 vanishingly unlikely already; the floor is a guard against
    that, not a resampling loop.
    """
    sampler = qmc.LatinHypercube(d=9, seed=seed)
    u = sampler.random(n=n)
    f_ede = np.maximum(_scale_uniform(u[:, 5], 0.0, 0.5), 1e-4)
    return {
        'ombh2':   _scale_uniform(   u[:, 0], 0.018,  0.028),
        'omch2':   _scale_uniform(   u[:, 1], 0.09,   0.15),
        'H0':      _scale_uniform(   u[:, 2], 60.0,   80.0),
        'ns':      _scale_uniform(   u[:, 3], 0.90,   1.10),
        'As':      _scale_loguniform(u[:, 4], 1.5e-9, 2.5e-9),
        'f_ede':   f_ede,
        'theta_i': _scale_uniform(   u[:, 6], 0.1,    3.1),
        'logzc':   _scale_uniform(   u[:, 7], 3.0,    4.3),
        'tau':     _lhc_tau(         u[:, 8]),
    }


print('LHC samplers defined.')

## 4. Set 1 — Planck ΛCDM  (`planck/`)

5500 realisations split into:
- **1a** (0–499): Planck 2018 MCMC chains (`base_plikHM_TTTEEE_lowl_lowE`), drawn directly with chain multiplicities as weights — all inter-parameter correlations respected automatically
- **1b** (500–5499): Broad ΛCDM via 6-D Latin Hypercube

Both components write to the same folder and CSV (with `component` column for filtering).

The six parameters extracted from the chains are:

| Parameter | Chain name | Note |
|---|---|---|
| H₀ | `H0` | derived parameter |
| Ω_b h² | `omegabh2` | sampled |
| Ω_c h² | `omegach2` | sampled |
| τ | `tau` | sampled |
| ln(10¹⁰Aₛ) | `logA` | sampled; converted via $A_s = e^{\rm logA} \times 10^{-10}$ |
| nₛ | `ns` | sampled |

$m_\nu = 0.06\,{\rm eV}$ and $\Omega_k = 0$ are fixed.

In [7]:
# ── Load Planck MCMC chains ────────────────────────────────────────────────────
_chain_root = os.path.join(
    '/nvme/h/lchristodoulou/data_p318',
    'COM_CosmoParams_base-plikHM-TTTEEE-lowl-lowE_R3',
    'base/plikHM_TTTEEE_lowl_lowE',
    'base_plikHM_TTTEEE_lowl_lowE',
)
_planck_chains = loadMCSamples(_chain_root, settings={'ignore_rows': 0.3})
_p = _planck_chains.getParams()
print(f'Planck chain samples after burn-in: {len(_planck_chains.samples):,}')

# ── Draw N_POSTERIOR_PLANCK rows without replacement, weighted by chain multiplicities ─
rng_p1  = np.random.default_rng(BASE_SEED + 10000)
_w_p    = _planck_chains.weights
_w_p    = _w_p / _w_p.sum()
_idx_p  = rng_p1.choice(len(_planck_chains.samples), size=N_POSTERIOR_PLANCK, replace=False, p=_w_p)

smp_p = {
    'H0':    _p.H0[_idx_p],
    'ombh2': _p.omegabh2[_idx_p],
    'omch2': _p.omegach2[_idx_p],
    'tau':   _p.tau[_idx_p],
    'lnAs':  _p.logA[_idx_p],
    'ns':    _p.ns[_idx_p],
}

print(f'\nSampled parameter ranges (N={N_POSTERIOR_PLANCK}):')
for name, arr in smp_p.items():
    print(f'  {name:8s}  mean={arr.mean():.5f}  std={arr.std():.5f}  '
          f'[{arr.min():.5f}, {arr.max():.5f}]')

tag_base          = f'planck_nside{nside}'
meta_path         = os.path.join(out_planck, f'metadata_planck_nside{nside}.csv')
fieldnames_planck = ['i', 'component', 'seed',
                     'H0', 'ombh2', 'omch2', 'tau', 'lnAs', 'As', 'ns', 'theta_star']

with open(meta_path, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames_planck).writeheader()

for i in tqdm(range(N_POSTERIOR_PLANCK), desc='Planck posterior (chains)'):
    seed_i = BASE_SEED + SEED_OFFSET['planck'] + i
    tau_i  = max(smp_p['tau'][i], 0.01)
    As_i   = np.exp(smp_p['lnAs'][i]) * 1e-10
    sp = generate_spectra(
        smp_p['H0'][i], smp_p['ombh2'][i], smp_p['omch2'][i], mnu, omk,
        tau=tau_i, As=As_i, ns=smp_p['ns'][i], lmax=2507,
    )
    cl_tt = dl_to_cl(sp.tt)
    cl_ee = dl_to_cl(sp.ee)
    cl_te = dl_to_cl(sp.te)
    cl_bb = dl_to_cl(sp.bb)
    make_and_save_maps(cl_tt, cl_ee, cl_bb, cl_te,
                       sim_mask_int, sim_mask_pol, nside, seed_i, fwhm_rad_sim,
                       out_planck, f'{tag_base}_{i:04d}')
    save_spectra(sp, cl_tt, cl_ee, cl_te, cl_bb,
                 out_planck, f'{tag_base}_{i:04d}')
    with open(meta_path, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=fieldnames_planck).writerow({
            'i': i, 'component': 'posterior', 'seed': seed_i,
            'H0': smp_p['H0'][i], 'ombh2': smp_p['ombh2'][i],
            'omch2': smp_p['omch2'][i], 'tau': tau_i,
            'lnAs': smp_p['lnAs'][i], 'As': As_i, 'ns': smp_p['ns'][i],
            'theta_star': sp.theta_star,
        })

print(f'Set 1a done: {N_POSTERIOR_PLANCK} realisations → {out_planck}/')

Planck chain samples after burn-in: 24,497

Sampled parameter ranges (N=500):
  H0        mean=67.29146  std=0.60297  [65.42236, 69.09912]
  ombh2     mean=0.02236  std=0.00015  [0.02191, 0.02277]
  omch2     mean=0.12016  std=0.00139  [0.11627, 0.12442]
  tau       mean=0.05440  std=0.00783  [0.03206, 0.08485]
  lnAs      mean=3.04455  std=0.01561  [2.99791, 3.10521]
  ns        mean=0.96498  std=0.00455  [0.94998, 0.97919]


Planck posterior (chains):   0%|          | 0/500 [00:00<?, ?it/s]

Set 1a done: 500 realisations → /nvme/h/lchristodoulou/data_p318/simulations/planck/


In [ ]:
# ── Set 1b: Planck broad (LHC) ────────────────────────────────────────────────
lhc1 = lhc_lcdm_6d(N_BROAD_PLANCK)

for j in tqdm(range(N_BROAD_PLANCK), desc='Planck broad LHC'):
    i      = N_POSTERIOR_PLANCK + j          # global index (500..5499)
    seed_i = BASE_SEED + SEED_OFFSET['planck'] + i
    As_i   = lhc1['As'][j]
    lnAs_i = np.log(As_i * 1e10)
    tau_i  = lhc1['tau'][j]
    sp = generate_spectra(
        lhc1['H0'][j], lhc1['ombh2'][j], lhc1['omch2'][j], mnu, omk,
        tau=tau_i, As=As_i, ns=lhc1['ns'][j], lmax=2507,
    )
    cl_tt = dl_to_cl(sp.tt)
    cl_ee = dl_to_cl(sp.ee)
    cl_te = dl_to_cl(sp.te)
    cl_bb = dl_to_cl(sp.bb)
    make_and_save_maps(cl_tt, cl_ee, cl_bb, cl_te,
                       sim_mask_int, sim_mask_pol, nside, seed_i, fwhm_rad_sim,
                       out_planck, f'{tag_base}_{i:04d}')
    save_spectra(sp, cl_tt, cl_ee, cl_te, cl_bb,
                 out_planck, f'{tag_base}_{i:04d}')
    with open(meta_path, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=fieldnames_planck).writerow({
            'i': i, 'component': 'broad', 'seed': seed_i,
            'H0': lhc1['H0'][j], 'ombh2': lhc1['ombh2'][j],
            'omch2': lhc1['omch2'][j], 'tau': tau_i,
            'lnAs': lnAs_i, 'As': As_i, 'ns': lhc1['ns'][j],
            'theta_star': sp.theta_star,
        })

print(f'Set 1b done: {N_BROAD_PLANCK} realisations → {out_planck}/')
print(f'Metadata → {meta_path}')

## 5. Set 2 — w0wa  (`w0wa/`)

1100 realisations split into:
- **2a** (0–99): DESI 2024 MCMC chains (`base_w_wa-desi-bao-all_planck2018-lowl-TT-clik_planck2018-lowl-EE-clik_planck-NPIPE-highl-CamSpec-TTTEEE`), drawn directly with chain multiplicities as weights — all inter-parameter correlations (including the $w_0$–$w_a$ anti-correlation) respected automatically
- **2b** (100–1099): Broad DE via 8-D Latin Hypercube

The eight parameters extracted from the chains are:

| Parameter | Chain column | Note |
|---|---|---|
| w₀ | `w` | CPL $w_0$ |
| wₐ | `wa` | CPL $w_a$ |
| H₀ | `H0` | derived |
| Ω_b h² | `ombh2` | sampled |
| Ω_c h² | `omch2` | sampled |
| τ | `tau` | sampled |
| ln(10¹⁰Aₛ) | `logA` | sampled; converted via $A_s = e^{\rm logA} \times 10^{-10}$ |
| nₛ | `ns` | sampled |

$m_\nu = 0.06\,{\rm eV}$ and $\Omega_k = 0$ are fixed.

In [ ]:
# ── INTERPRETATION CEILING: posterior arm vs matched arm ─────────────────────
# The survey-posterior sets (Planck chain vs DESI w0wa chain) are a VALID
# experiment: "do the two survey-preferred cosmologies produce distinguishable
# skies?" theta* is auto-matched (both chains include Planck CMB), so geometry is
# NOT a confound. sigma8 is NOT matched, but it acts only through lensing.
#   * USE_LENSED_SPECTRA=False -> sigma8 is inert. No confound; separation (if any)
#     is base-parameter/primary-spectrum only. Safe to interpret as-is.
#   * USE_LENSED_SPECTRA=True  -> the DESI arm differs in sigma8 (lensing
#     amplitude), reproducible inside LCDM by tuning As. A lensed separation
#     therefore CANNOT be read as "dynamical dark energy is detectable" — it may
#     just be a sigma8 offset riding along with the DESI fit. That stronger claim
#     needs the sigma8/As/tau-matched arm (see [2] above).
# TL;DR: posterior arm answers "are these two cosmologies distinguishable";
#        only the matched arm answers "is the DE sector itself the discriminator".
# ─────────────────────────────────────────────────────────────────────────────

# [2] DEFERRABLE — only matters once USE_LENSED_SPECTRA=True. The controlled
#     "matched" arm (LCDM vs w0wa differing ONLY in the DE sector). Not needed
#     while lensing is off; the confounds it removes are lensing-borne.
#     When building it:
#       (a) theta* ordering. To hold the acoustic scale fixed, pass
#           thetastar=... (NOT H0=, and NOT cosmomc_theta which is LCDM-calibrated)
#           and let CAMB solve H0. The DE model MUST be set in the SAME set_params
#           call as thetastar, e.g.
#               camb.set_params(thetastar=..., dark_energy_model='ppf', w=w0, wa=wa,
#                               ombh2=..., omch2=..., ...)
#           Solving H0 first and attaching params.DarkEnergy afterward leaves
#           theta* drifted (~0.4 sigma_Planck for a mild w0wa, larger further from
#           LCDM) — a coherent, class-correlated leak a network will grab.
#       (b) sigma8 / lensing-amplitude match. At fixed theta*, w0wa still gives a
#           different sigma8 -> different lensing amplitude. sigma8 ∝ sqrt(As)
#           exactly, so match it in one shot:
#               As_new  = As_ref * (sigma8_ref / sigma8_w0wa)**2
#           But As also sets the primary amplitude (∝ As*exp(-2*tau)), so restore
#           it by co-moving tau:
#               tau_new = tau_ref + 0.5*log(As_new / As_ref)
#           Two knobs, two constraints, closed form (no iteration). Guard tau to a
#           physical range (>~0.03); large broad-LHC w0wa corners need big As
#           rescales and can push tau out of bounds — drop/flag those samples.
#           Caveat: exp(-2tau) is clean only at ell > ~20; tau leaves a low-ell/EE
#           reionization footprint. Fine for temperature-only above ell~30.
#           Alt: leave As/tau alone and rescale lensing directly with
#           Alens = (sigma8_ref/sigma8_w0wa)**2 (surgical but phenomenological).
#
# [3] nside=256 caps ell_max at 3*nside-1 = 767. The lensing peak-smoothing signal
#     lives mostly at ell > 1000, so even with lensing ON the non-Gaussian info a
#     spherical CNN can use is thin here. Go to nside 512 (ell_max 1535) or 1024
#     if the lensing channel is the thing under test.

In [7]:
# ── Load DESI MCMC chains ─────────────────────────────────────────────────────
_desi_chain_dir = (
    '/nvme/h/lchristodoulou/data_p318/'
    'base_w_wa-desi-bao-all_planck2018-lowl-TT-clik_planck2018-lowl-EE-clik_planck-NPIPE-highl-CamSpec-TTTEEE'
)
_desi_chains = loadMCSamples(
    os.path.join(_desi_chain_dir, 'chain'),
    settings={'ignore_rows': 0.3},
)
_pd = _desi_chains.getParams()
print(f'DESI chain samples after burn-in: {len(_desi_chains.samples):,}')

# ── Draw N_POSTERIOR_W0WA rows without replacement, weighted by chain multiplicities ─
rng_d   = np.random.default_rng(BASE_SEED + 20000)
_w_d    = _desi_chains.weights
_w_d    = _w_d / _w_d.sum()
_idx_d  = rng_d.choice(len(_desi_chains.samples), size=N_POSTERIOR_W0WA, replace=False, p=_w_d)

w0_post = _pd.w[_idx_d]
wa_post = _pd.wa[_idx_d]
smp_d = {
    'H0':    _pd.H0[_idx_d],
    'ombh2': _pd.ombh2[_idx_d],
    'omch2': _pd.omch2[_idx_d],
    'tau':   _pd.tau[_idx_d],
    'lnAs':  _pd.logA[_idx_d],
    'ns':    _pd.ns[_idx_d],
}

print(f'\nSampled parameter ranges (N={N_POSTERIOR_W0WA}):')
for name, arr in [*smp_d.items(), ('w0', w0_post), ('wa', wa_post)]:
    print(f'  {name:8s}  mean={arr.mean():.5f}  std={arr.std():.5f}  '
          f'[{arr.min():.5f}, {arr.max():.5f}]')

tag_base2   = f'w0wa_nside{nside}'
meta_path2  = os.path.join(out_w0wa, f'metadata_w0wa_nside{nside}.csv')
fieldnames_w0wa = ['i', 'component', 'seed',
                   'H0', 'ombh2', 'omch2', 'tau', 'lnAs', 'As', 'ns',
                   'w0', 'wa', 'theta_star']

with open(meta_path2, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames_w0wa).writeheader()

for i in tqdm(range(N_POSTERIOR_W0WA), desc='w0wa DESI posterior (chains)'):
    seed_i = BASE_SEED + SEED_OFFSET['w0wa'] + i
    tau_i  = max(smp_d['tau'][i], 0.01)
    As_i   = np.exp(smp_d['lnAs'][i]) * 1e-10
    sp = generate_spectra(
        smp_d['H0'][i], smp_d['ombh2'][i], smp_d['omch2'][i], mnu, omk,
        tau=tau_i, As=As_i, ns=smp_d['ns'][i], lmax=2507,
        w0=w0_post[i], wa=wa_post[i],
    )
    cl_tt = dl_to_cl(sp.tt)
    cl_ee = dl_to_cl(sp.ee)
    cl_te = dl_to_cl(sp.te)
    cl_bb = dl_to_cl(sp.bb)
    make_and_save_maps(cl_tt, cl_ee, cl_bb, cl_te,
                       sim_mask_int, sim_mask_pol, nside, seed_i, fwhm_rad_sim,
                       out_w0wa, f'{tag_base2}_{i:04d}')
    save_spectra(sp, cl_tt, cl_ee, cl_te, cl_bb,
                 out_w0wa, f'{tag_base2}_{i:04d}')
    with open(meta_path2, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=fieldnames_w0wa).writerow({
            'i': i, 'component': 'posterior', 'seed': seed_i,
            'H0': smp_d['H0'][i], 'ombh2': smp_d['ombh2'][i],
            'omch2': smp_d['omch2'][i], 'tau': tau_i,
            'lnAs': smp_d['lnAs'][i], 'As': As_i, 'ns': smp_d['ns'][i],
            'w0': w0_post[i], 'wa': wa_post[i], 'theta_star': sp.theta_star,
        })

print(f'Set 2a done: {N_POSTERIOR_W0WA} realisations → {out_w0wa}/')

DESI chain samples after burn-in: 63,629

Sampled parameter ranges (N=500):
  H0        mean=63.62907  std=1.86134  [58.72046, 69.77200]
  ombh2     mean=0.02220  std=0.00012  [0.02185, 0.02258]
  omch2     mean=0.11954  std=0.00091  [0.11671, 0.12237]
  tau       mean=0.05148  std=0.00766  [0.02383, 0.07498]
  lnAs      mean=3.03457  std=0.01601  [2.97713, 3.08132]
  ns        mean=0.96412  std=0.00347  [0.95328, 0.97291]
  w0        mean=-0.42331  std=0.20945  [-1.08648, 0.07688]
  wa        mean=-1.72715  std=0.60640  [-2.99996, 0.16319]


w0wa DESI posterior (chains):   0%|          | 0/500 [00:00<?, ?it/s]

Set 2a done: 500 realisations → /nvme/h/lchristodoulou/data_p318/simulations/w0wa/


In [ ]:
# ── Set 2b: w0wa broad (LHC) ──────────────────────────────────────────────────
lhc2 = lhc_w0wa_8d(N_BROAD_W0WA)

for j in tqdm(range(N_BROAD_W0WA), desc='w0wa broad LHC'):
    i      = N_POSTERIOR_W0WA + j
    seed_i = BASE_SEED + SEED_OFFSET['w0wa'] + i
    As_i   = lhc2['As'][j]
    lnAs_i = np.log(As_i * 1e10)
    tau_i  = lhc2['tau'][j]
    sp = generate_spectra(
        lhc2['H0'][j], lhc2['ombh2'][j], lhc2['omch2'][j], mnu, omk,
        tau=tau_i, As=As_i, ns=lhc2['ns'][j], lmax=2507,
        w0=lhc2['w0'][j], wa=lhc2['wa'][j],
    )
    cl_tt = dl_to_cl(sp.tt)
    cl_ee = dl_to_cl(sp.ee)
    cl_te = dl_to_cl(sp.te)
    cl_bb = dl_to_cl(sp.bb)
    make_and_save_maps(cl_tt, cl_ee, cl_bb, cl_te,
                       sim_mask_int, sim_mask_pol, nside, seed_i, fwhm_rad_sim,
                       out_w0wa, f'{tag_base2}_{i:04d}')
    save_spectra(sp, cl_tt, cl_ee, cl_te, cl_bb,
                 out_w0wa, f'{tag_base2}_{i:04d}')
    with open(meta_path2, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=fieldnames_w0wa).writerow({
            'i': i, 'component': 'broad', 'seed': seed_i,
            'H0': lhc2['H0'][j], 'ombh2': lhc2['ombh2'][j],
            'omch2': lhc2['omch2'][j], 'tau': tau_i,
            'lnAs': lnAs_i, 'As': As_i, 'ns': lhc2['ns'][j],
            'w0': lhc2['w0'][j], 'wa': lhc2['wa'][j], 'theta_star': sp.theta_star,
        })

print(f'Set 2b done: {N_BROAD_W0WA} realisations → {out_w0wa}/')
print(f'Metadata → {meta_path2}')

## 6. Set 3 — Oscillations  (`oscillations/`)

1000 realisations, all broad LHC (no posterior component).

Primordial power spectrum:
$$P(k) = A_s\, k^{n_s-1}\left[1 + A_{\rm lin}\sin\!\left(\phi + {\rm freq}\cdot\ln(k/k_0)\right)\right]$$
with $k_0 = 0.06$, width = 0.04 fixed.

| Extra param | Range | Sampling |
|-------------|-------|----------|
| A_lin | [0.01, 0.06] | LHC uniform |
| freq | [20, 40] | LHC uniform (float) |
| phase φ | [0, 2π] | LHC uniform |

For Phase 1 the oscillation frequency is kept low (`freq ∈ [20, 40]`), placing features at
Δℓ ~ 50–100 across the first three acoustic peaks (ℓ ~ 200–500) — the cleanly-resolved,
discriminable regime. H₀ is fixed at the Planck best-fit for this set.

In [ ]:
# ── Set 3: Oscillations broad (LHC) ──────────────────────────────────────────
# H0 fixed at Planck best-fit; w0/wa = ΛCDM. τ gated by INCLUDE_POLARIZATION
# (fixed if off, independent LHC draw if on — see _lhc_tau).
H0_osc = 67.4

lhc3 = lhc_osc_8d(N_OSC)

tag_base3   = f'oscillations_nside{nside}'
meta_path3  = os.path.join(out_osc, f'metadata_oscillations_nside{nside}.csv')
fieldnames_osc = ['i', 'component', 'seed',
                  'H0', 'ombh2', 'omch2', 'tau', 'lnAs', 'As', 'ns',
                  'A_lin', 'freq', 'phase', 'theta_star']

with open(meta_path3, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames_osc).writeheader()

for i in tqdm(range(N_OSC), desc='Oscillations broad LHC'):
    seed_i  = BASE_SEED + SEED_OFFSET['oscillations'] + i
    As_i    = lhc3['As'][i]
    lnAs_i  = np.log(As_i * 1e10)
    tau_i   = lhc3['tau'][i]
    freq_i  = float(lhc3['freq'][i])
    phase_i = float(lhc3['phase'][i])
    sp = generate_spectra(
        H0_osc, lhc3['ombh2'][i], lhc3['omch2'][i], mnu, omk,
        tau=tau_i, As=As_i, ns=lhc3['ns'][i], lmax=2507,
        custom_PK=True, amp=lhc3['A_lin'][i], freq=freq_i,
        wid=0.04, centre=0.06, phase=phase_i,
    )
    cl_tt = dl_to_cl(sp.tt)
    cl_ee = dl_to_cl(sp.ee)
    cl_te = dl_to_cl(sp.te)
    cl_bb = dl_to_cl(sp.bb)
    make_and_save_maps(cl_tt, cl_ee, cl_bb, cl_te,
                       sim_mask_int, sim_mask_pol, nside, seed_i, fwhm_rad_sim,
                       out_osc, f'{tag_base3}_{i:04d}')
    save_spectra(sp, cl_tt, cl_ee, cl_te, cl_bb,
                 out_osc, f'{tag_base3}_{i:04d}')
    with open(meta_path3, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=fieldnames_osc).writerow({
            'i': i, 'component': 'broad', 'seed': seed_i,
            'H0': H0_osc, 'ombh2': lhc3['ombh2'][i],
            'omch2': lhc3['omch2'][i], 'tau': tau_i,
            'lnAs': lnAs_i, 'As': As_i, 'ns': lhc3['ns'][i],
            'A_lin': lhc3['A_lin'][i], 'freq': freq_i, 'phase': phase_i,
            'theta_star': sp.theta_star,
        })

print(f'Set 3 done: {N_OSC} realisations → {out_osc}/')
print(f'Metadata → {meta_path3}')

## 7. Set 4 — Early Dark Energy  (`ede/`)

N_EDE realisations, all broad LHC (no posterior component — no MCMC chain with
this DE extension is loaded here, unlike Sets 1/2).

CAMB's `camb.dark_energy.AxionEffectiveFluid` (Poulin, Smith, Karwal &
Kamionkowski 2018, arXiv:1806.10608) models a rolling axion-like scalar with
potential $V(\phi)\propto[1-\cos(\phi/f)]^n$ that behaves like a cosmological
constant until $z_c$, then dilutes as a fluid with equation of state $w_n$ —
it replaces the DE sector entirely (no separate w0/wa alongside it).

| Extra param | Range | Sampling |
|-------------|-------|----------|
| f_EDE (`fde_zc`) | [0, 0.5]† | LHC uniform |
| theta_i | [0.1, 3.1] | LHC uniform |
| log10(zc) | [3, 4.3] | LHC uniform (`zc` passed to CAMB) |
| w_n | 0.5 — **fixed**, not sampled | see note below |

†floored at 1e-4, not the literal 0.0 — CAMB hard-crashes (native abort) at
`fde_zc=0.0` exactly.

**Why w_n isn't sampled**: w_n is not a free nuisance parameter of the fluid —
it's fixed by the axion potential's power index n via w_n=(n-1)/(n+1). n=3
(w_n=1/2, `EDE_W_N`) is the near-universal benchmark in the EDE literature: it's
the value that lets the field dilute fast enough after zc to resolve the H0
tension without overshooting other observables, and it's what essentially every
major EDE constraint paper (Poulin et al. 2018; Smith et al. 2020; Hill et al.
2020) reports for. Treating it as a free continuous LHC dimension would mix
physically distinct potential shapes into a single axis with no literature
prior to draw from — so it's fixed, exactly like `n=3` is fixed in those
analyses, rather than sampled.

Unlike Set 3 (`oscillations`), H0 is left **free** here rather than fixed at the
Planck best-fit: EDE's headline feature is precisely that raising f_EDE lets H0
float higher while still matching theta_star, so fixing H0 would hide the
degeneracy this set exists to let a network learn.

In [ ]:
# ── Set 4: Early Dark Energy broad (LHC) ─────────────────────────────────────
# No MCMC posterior arm (no Planck/DESI chain with this DE extension available
# here) — broad LHC only, like Set 3. w_n fixed at EDE_W_N (see markdown above).
lhc4 = lhc_ede_9d(N_EDE)

tag_base4   = f'ede_nside{nside}'
meta_path4  = os.path.join(out_ede, f'metadata_ede_nside{nside}.csv')
fieldnames_ede = ['i', 'component', 'seed',
                  'H0', 'ombh2', 'omch2', 'tau', 'lnAs', 'As', 'ns',
                  'f_ede', 'theta_i', 'zc', 'logzc', 'w_n', 'theta_star']

with open(meta_path4, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=fieldnames_ede).writeheader()

for i in tqdm(range(N_EDE), desc='EDE broad LHC'):
    seed_i    = BASE_SEED + SEED_OFFSET['ede'] + i
    As_i      = lhc4['As'][i]
    lnAs_i    = np.log(As_i * 1e10)
    tau_i     = lhc4['tau'][i]
    f_ede_i   = float(lhc4['f_ede'][i])
    theta_i_i = float(lhc4['theta_i'][i])
    logzc_i   = float(lhc4['logzc'][i])
    zc_i      = 10 ** logzc_i
    sp = generate_spectra(
        lhc4['H0'][i], lhc4['ombh2'][i], lhc4['omch2'][i], mnu, omk,
        tau=tau_i, As=As_i, ns=lhc4['ns'][i], lmax=2507,
        ede_params=(EDE_W_N, f_ede_i, zc_i, theta_i_i),
    )
    cl_tt = dl_to_cl(sp.tt)
    cl_ee = dl_to_cl(sp.ee)
    cl_te = dl_to_cl(sp.te)
    cl_bb = dl_to_cl(sp.bb)
    make_and_save_maps(cl_tt, cl_ee, cl_bb, cl_te,
                       sim_mask_int, sim_mask_pol, nside, seed_i, fwhm_rad_sim,
                       out_ede, f'{tag_base4}_{i:04d}')
    save_spectra(sp, cl_tt, cl_ee, cl_te, cl_bb,
                 out_ede, f'{tag_base4}_{i:04d}')
    with open(meta_path4, 'a', newline='') as f:
        csv.DictWriter(f, fieldnames=fieldnames_ede).writerow({
            'i': i, 'component': 'broad', 'seed': seed_i,
            'H0': lhc4['H0'][i], 'ombh2': lhc4['ombh2'][i],
            'omch2': lhc4['omch2'][i], 'tau': tau_i,
            'lnAs': lnAs_i, 'As': As_i, 'ns': lhc4['ns'][i],
            'f_ede': f_ede_i, 'theta_i': theta_i_i, 'zc': zc_i,
            'logzc': logzc_i, 'w_n': EDE_W_N, 'theta_star': sp.theta_star,
        })

print(f'Set 4 done: {N_EDE} realisations → {out_ede}/')
print(f'Metadata → {meta_path4}')

## 8. Sanity Checks

Run after a small test batch (`N_POSTERIOR_PLANCK=2, N_POSTERIOR_W0WA=1, N_BROAD_PLANCK=3, N_BROAD_W0WA=3, N_OSC=3, N_EDE=3`). Checks:
1. File inventory — correct number of FITS + CSV per set
2. Map properties — nside, dtype, no NaN/Inf
3. Statistics — mean ≈ 0, std in expected range
4. E/B ratio — ⟨Cℓ_EE⟩ / ⟨Cℓ_BB⟩ ≫ 1 (only if INCLUDE_POLARIZATION)
5. Metadata integrity — row count, unique seeds, `component` values

In [ ]:
import pandas as pd
from pathlib import Path

_sets = {
    'planck':       {'folder': out_planck, 'tag': f'planck_nside{nside}',
                     'meta': f'metadata_planck_nside{nside}.csv',
                     'total': N_POSTERIOR_PLANCK + N_BROAD_PLANCK},
    'w0wa':         {'folder': out_w0wa,   'tag': f'w0wa_nside{nside}',
                     'meta': f'metadata_w0wa_nside{nside}.csv',
                     'total': N_POSTERIOR_W0WA + N_BROAD_W0WA},
    'oscillations': {'folder': out_osc,    'tag': f'oscillations_nside{nside}',
                     'meta': f'metadata_oscillations_nside{nside}.csv',
                     'total': N_OSC},
    'ede':          {'folder': out_ede,    'tag': f'ede_nside{nside}',
                     'meta': f'metadata_ede_nside{nside}.csv',
                     'total': N_EDE},
}
_comps_expected = ['T', 'Q', 'U', 'E', 'B'] if INCLUDE_POLARIZATION else ['T']
_fsky_int = sim_mask_int.mean()
_fsky_pol = sim_mask_pol.mean()

# ── 1. File inventory ─────────────────────────────────────────────────────────
print('── 1. File inventory ──────────────────────────────────────────────────')
for name, info in _sets.items():
    n_fits = len(list(Path(info['folder']).glob('*.fits')))
    n_csv  = len(list(Path(info['folder']).glob('*.csv')))
    expected = info['total'] * len(_comps_expected)
    ok = '✓' if n_fits == expected else f'✗ (expected {expected})'
    print(f"  {name:15s}  {n_fits:5d} FITS {ok}   {n_csv} CSV")

# ── 2 & 3. Map properties and statistics (realisation 0000) ──────────────────
print('\n── 2 & 3. Map properties / statistics (realisation 0000) ─────────────')
for name, info in _sets.items():
    T_path = os.path.join(info['folder'], f"T_{info['tag']}_0000.fits")
    if not os.path.exists(T_path):
        print(f'  {name}: T_0000.fits not found — skipping'); continue
    T = hp.read_map(T_path, verbose=False)
    nz = T != 0
    print(f"  {name}: nside={hp.get_nside(T)} "
          f"float32={'✓' if T.dtype==np.float32 else '✗'} "
          f"NaN-free={'✓' if not np.isnan(T).any() else '✗'} "
          f"f_sky={nz.mean():.3f}  "
          f"T std={T[nz].std():.1f} μK")

# ── 4. E/B ratio ──────────────────────────────────────────────────────────────
if INCLUDE_POLARIZATION:
    print('\n── 4. E/B power ratio (ℓ = 50–500, realisation 0000) ─────────────────')
    for name, info in _sets.items():
        E_path = os.path.join(info['folder'], f"E_{info['tag']}_0000.fits")
        B_path = os.path.join(info['folder'], f"B_{info['tag']}_0000.fits")
        if not (os.path.exists(E_path) and os.path.exists(B_path)):
            print(f'  {name}: E/B maps not found — skipping'); continue
        cl_E = hp.anafast(hp.read_map(E_path, verbose=False), lmax=500)[50:]
        cl_B = hp.anafast(hp.read_map(B_path, verbose=False), lmax=500)[50:]
        ratio = cl_E.mean() / max(cl_B.mean(), 1e-30)
        print(f"  {name:15s}  ⟨Cl_EE⟩/⟨Cl_BB⟩ = {ratio:6.1f}  "
              f"{'✓' if ratio > 5 else '⚠ unexpectedly low'}")

# ── 5. Metadata integrity ─────────────────────────────────────────────────────
print('\n── 5. Metadata CSV ────────────────────────────────────────────────────')
for name, info in _sets.items():
    csv_path = os.path.join(info['folder'], info['meta'])
    if not os.path.exists(csv_path):
        print(f'  {name}: CSV not found'); continue
    df = pd.read_csv(csv_path)
    seeds_ok = df['seed'].nunique() == len(df)
    comps_ok = set(df['component'].unique()) <= {'posterior', 'broad'}
    count_ok = len(df) == info['total']
    print(f"  {name:15s}  {len(df)} rows {'✓' if count_ok else '✗'}  "
          f"unique seeds {'✓' if seeds_ok else '✗'}  "
          f"components {df['component'].value_counts().to_dict()}  "
          f"cols: {list(df.columns)}")